In [1]:
import json
import os
import re
import yaml
from google.cloud import bigquery, storage

import pandas as pd
import google.auth
from dotenv import load_dotenv
from tqdm.auto import tqdm

import vertexai
from vertexai.generative_models import (
    GenerativeModel,
    GenerationConfig,
    HarmCategory,
    HarmBlockThreshold,
    GenerationResponse,
)
from asynciolimiter import Limiter
from tqdm.asyncio import tqdm_asyncio
import jsonlines

from transformers import AutoTokenizer

import asyncio
import nest_asyncio

nest_asyncio.apply()
load_dotenv()

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


True

In [2]:
# CONFIG Bucket
CONFIG_BUCKET = "tmhcc-dev-intermediate"

credentials, project_id = google.auth.default()
LOCATION = os.getenv("GCP_LOCATION")

vertexai.init(
    project=os.getenv("GCP_PROJECT"), location=LOCATION, credentials=credentials
)

storage_client = storage.Client()
bigquery_client = bigquery.Client()

BASE_FOLDER = os.path.join("..", "data", "info_collection")
ROOT_DATA_DIR = os.path.join("..", "..", "..", *os.getenv("DATA_DIR").split(","))
os.path.exists(ROOT_DATA_DIR)

True

## Re-writing Problems-Solutions

In [3]:
async def get_raw_output_from_llm(
        df:pd.DataFrame,
        interim_results_df:pd.DataFrame,
        interim_fname:str,
        system_instruction:str,
        model_name:str='gemini-1.0-pro-002',
        prompt_col:str='Prompt',
        save_every:int=100,
        rpm:int=200,
    ):

    ## Model Setup ##
    # Safety Settings
    safety_settings = {
        # HarmCategory.HARM_CATEGORY_DANGEROUS: HarmBlockThreshold.BLOCK_NONE,
        HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_ONLY_HIGH,
        # HarmCategory.HARM_CATEGORY_DEROGATORY: HarmBlockThreshold.BLOCK_NONE,
        HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_ONLY_HIGH,
        HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_ONLY_HIGH,
        # HarmCategory.HARM_CATEGORY_MEDICAL: HarmBlockThreshold.BLOCK_NONE,
        # HarmCategory.HARM_CATEGORY_SEXUAL: HarmBlockThreshold.BLOCK_NONE,
        HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_ONLY_HIGH,
        # HarmCategory.HARM_CATEGORY_TOXICITY: HarmBlockThreshold.BLOCK_NONE,
        # HarmCategory.HARM_CATEGORY_UNSPECIFIED: HarmBlockThreshold.BLOCK_NONE,
        # HarmCategory.HARM_CATEGORY_VIOLENCE: HarmBlockThreshold.BLOCK_NONE,
    }

    # Generation Config
    generation_config = GenerationConfig(
        candidate_count=1,
        stop_sequences=[],
        max_output_tokens=8192,
        temperature=1,
        top_p=1,
        top_k=40,
    )

    # Model
    gemini_model = GenerativeModel(
        model_name=model_name,
        safety_settings=safety_settings,
        generation_config=generation_config,
        system_instruction=system_instruction,
    )

    # Setup Limiter
    # max_concurrent_tasks = 60
    rate_limiter = Limiter(rpm // 60)

    ## Async Function Call to Model ##
    async def gemini_async_call(text:str, idx:int, id:str):
        # with semaphore:
        await rate_limiter.wait()
        try:
            return await gemini_model.generate_content_async([text]), int(idx), id
        except Exception:
            return None, int(idx), id

    ## Main Function ##
    tasks = []
    # too_long_counter = 0
    interim_results = []
    for idx in tqdm(df[df[prompt_col].notna()].index, desc='Creating Tasks'):

        if idx in interim_results_df.id.values:
            continue
        
        input_prompt = df[prompt_col][idx]
        exp_id = df['id'][idx]
        tasks.append(asyncio.create_task(gemini_async_call(input_prompt, idx, exp_id)))

    # print(f"Too Long Counter: {too_long_counter}")
    print(f'{len(tasks)} Tasks Created')
    counter = 0
    safety_counter = 0
    input_not_processed_counter = 0
    for done in tqdm_asyncio.as_completed(tasks, ncols=100, desc="Waiting for Tasks:", leave=True):

        try:

            result, idx, exp_id  = await done
            result: GenerationResponse
            idx: int
            counter += 1

            if result is None:
                raise Exception

            try:
                interim_results.append({
                    'id': idx,
                    'exp_id': exp_id,
                    'is_blocked': result.to_dict()['candidates'][0]['finish_reason'] != 1 or result.to_dict()['candidates'][0]['finish_reason'] != 'STOP',
                    'text': result.text,
                    'input_tokens': result._raw_response.usage_metadata.prompt_token_count,
                    'output_tokens': result._raw_response.usage_metadata.candidates_token_count
                })
                
            except Exception:
                safety_counter += 1
                interim_results.append({
                    'id': idx,
                    'exp_id': exp_id,
                    'is_blocked': True,
                    'text': None,
                    'input_tokens': None,
                    'output_tokens': None,
                })
                continue
        
        except Exception:
            # print(f"Error at index {idx} - input not processed")
            input_not_processed_counter += 1
            interim_results.append({
                    'id': idx,
                    'exp_id': exp_id,
                    'is_blocked': False,
                    'text': None,
                    'input_tokens': None,
                    'output_tokens': None,
                    'drop_reason': 'Input not processed'
                })
        
        # Save to file
        if counter % save_every == 0:
            with jsonlines.open(interim_fname, 'a') as writer:
                writer.write_all(interim_results)
            interim_results = []

    print(f"Number of Safety Blocks: {safety_counter}")
    print(f"Number of Input not processed: {input_not_processed_counter}")
    
    with jsonlines.open(interim_fname, 'a') as writer:
        writer.write_all(interim_results)
        
    return df

In [159]:
# interim_fname = f'interim_results_Preprocessing_input_data_{pd.to_datetime("now").strftime("%Y%m%d_%H%M%S")}.jsonl'
interim_fname = 'interim_results_Preprocessing_input_data_gemini_flash.jsonl'
print(f"Interim Results File: {interim_fname}")

if not os.path.exists(interim_fname):
    with jsonlines.open(interim_fname, 'w') as writer:
        writer.write_all([])

Interim Results File: interim_results_Preprocessing_input_data_gemini_flash.jsonl


In [186]:
interim_fname = 'interim_results_Preprocessing_input_data_gemini_flash.jsonl'
with jsonlines.open(interim_fname, 'r') as reader:
    interim_results_df = pd.DataFrame(reader)

if len(interim_results_df) == 0:
    interim_results_df = pd.DataFrame(columns=['id', 'is_blocked', 'text', 'input_tokens', 'output_tokens', 'DropReason'])

In [187]:
interim_results_df.text.isna().sum()

23

In [188]:
interim_results_df = interim_results_df[interim_results_df.text.notna()].copy(deep=True)
interim_results_df.to_json(interim_fname, orient='records', lines=True, index=False)

In [183]:
input_df = pd.read_json(os.path.join(BASE_FOLDER, 'preprocess_input.jsonl'), lines=True)
input_df

,id,problem,solution,prompt
0,Exp0000,Write the next three natural numbers after 10999.,"10,999+1 = 11,000\n11,000 + 1 = 11,001\n11,001...",<math_problem>\nWrite the next three natural n...
1,Exp0001,Write the three whole numbers occurring just b...,Three whole numbers occurring just before 1000...,<math_problem>\nWrite the three whole numbers ...
2,Exp0002,Which is the smallest whole number?,0 (zero) is the smallest whole number. All the...,<math_problem>\nWhich is the smallest whole nu...
3,Exp0003,How many whole numbers are there between 32 an...,Whole numbers between 32 and 53 = 20 (53 − 32 ...,<math_problem>\nHow many whole numbers are the...
4,Exp0004,Write the successor of 2440701.,The successor of 2440701 is\n2440701 + 1 = 244...,<math_problem>\nWrite the successor of 2440701...
...,...,...,...,...
4321,Exp4321,If each element of a second order determinant ...,The total number of determinants of second ord...,<math_problem>\nIf each element of a second or...
4322,Exp4322,An electronic assembly consists of two subsyst...,Let the event in which A fails and B fails be ...,<math_problem>\nAn electronic assembly consist...
4323,Exp4323,If A and B are two events such that P (A) ≠ 0 ...,"P (A) ≠ 0 and P(B|A) = 1\n\nThus, the correct ...",<math_problem>\nIf A and B are two events such...
4324,Exp4324,"If P (A|B) > P (A), then which of the followin...","\nThus, the correct answer is C.","<math_problem>\nIf P (A|B) > P (A), then which..."


In [184]:
system_instruction = r"""You are a math professor who is very rigorous in writing down solutions. You also care about your students and write solution walkthrough in step-by-step manner and using latex everywhere necessary.

You will be given a math problem and its solution, you job is to re-write the problem in good latex and solution in very step-by-step manner for your students.
If you can find an efficient solution than given, please give that solution.

The output has to be in the YAML format:
```
to_drop: BOOLEAN
problem: "PROBLEM RE-WRITTEN IN PROPER LATEX"
solution:
  - "SOLUTION RE-WRITTEN IN PROPER LATEX IN MULTIPLE STEPS AS A LIST"
```
You should drop a problem altogether if its incomplete or problem has includes a diagram or is about constructing one or the problem is MCQ.
Keep the number of steps as minimum as possible.

Study the given solution, if you can solve the problem with a more efficient solution give that.
Combine latex commands in single step, not multiple.

DO NOT USE `\begin{align*}` `\begin{align}` `\begin{aligned}` or any kind of other alignment in latex code.
Take care that you use right YAML format, do not nest steps!
"""
print(system_instruction)

You are a math professor who is very rigorous in writing down solutions. You also care about your students and write solution walkthrough in step-by-step manner and using latex everywhere necessary.

You will be given a math problem and its solution, you job is to re-write the problem in good latex and solution in very step-by-step manner for your students.
If you can find an efficient solution than given, please give that solution.

The output has to be in the YAML format:
```
to_drop: BOOLEAN
problem: "PROBLEM RE-WRITTEN IN PROPER LATEX"
solution:
  - "SOLUTION RE-WRITTEN IN PROPER LATEX IN MULTIPLE STEPS AS A LIST"
```
You should drop a problem altogether if its incomplete or problem has includes a diagram or is about constructing one or the problem is MCQ.
Keep the number of steps as minimum as possible.

Study the given solution, if you can solve the problem with a more efficient solution give that.
Combine latex commands in single step, not multiple.

DO NOT USE `\begin{align*}` 

In [185]:
# intermedate_df = asyncio.run(get_raw_output_from_llm_v2(chucked_df, interim_results_df))
intermedate_df = asyncio.run(
    get_raw_output_from_llm(
        df=input_df,
        interim_results_df=interim_results_df,
        interim_fname=interim_fname,
        system_instruction=system_instruction,
        model_name='gemini-1.5-flash-preview-0514',
        # model_name='gemini-1.0-pro-002',
        prompt_col='prompt',
        save_every=10,
        rpm=180,
    )
)

Creating Tasks:   0%|          | 0/4326 [00:00<?, ?it/s]

31 Tasks Created


Waiting for Tasks:: 100%|███████████████████████████████████████████| 31/31 [00:16<00:00,  1.91it/s]

Number of Safety Blocks: 23
Number of Input not processed: 0


### Attempts to load

In [88]:
json.loads("```json\n{\n    \"problem\": \"Is the statement \\\\\\\"Zero is the smallest natural number.\\\\\\\" true or false?\",\n    \"solution\": [\n        \"The statement is **False**.\",\n        \"The natural numbers are the counting numbers: 1, 2, 3, 4, ....\",\n        \"Therefore, zero is not a natural number. The smallest natural number is 1.\"\n    ],\n    \"to_drop\": false\n}\n```".replace('```json\n', '').replace('```', ''))

{'problem': 'Is the statement \\"Zero is the smallest natural number.\\" true or false?',
 'solution': ['The statement is **False**.',
  'The natural numbers are the counting numbers: 1, 2, 3, 4, ....',
  'Therefore, zero is not a natural number. The smallest natural number is 1.'],
 'to_drop': False}

In [197]:
with open(interim_fname, 'r') as f:
    raw_data = f.readlines()

# json.loads(json.loads(raw_data[22])['text'].replace('```json\n', '').replace('```', ''))
text = json.loads(raw_data[22])['text'].replace('```yaml\n', '').replace('```', '').replace('\\\\newline', ' ')
text

'to_drop: false\nproblem:  "Is the following statement true or false?   The predecessor of a two-digit number is never a single-digit number."\nsolution:\n  - "The statement is **False**.  The predecessor of $10$ is $9$ and $9$ is a single-digit number." \n'

In [328]:
text = 'to_drop: false\nproblem: "Is the following statement true or false? \nZero is the smallest natural number".\nsolution:\n  - "The given statement is **False**. \nThe set of natural numbers is defined as \n$$\\mathbb{N} = \\{ 1, 2, 3, \\dots \\}$$. \nTherefore, 1 is the smallest natural number."\n'

# x, y = re.search(r'to_drop: (.*)\n', text).span()
# print(text[x:y-1], '\n')

# x, y = re.search(r'problem: (.*)\nsol', text, re.DOTALL).span()
# print(text[x:y-4], '\n')

# x, y = re.search(r'solution:\n (.*)', text, re.DOTALL).span()
# print(text[x:y], '\n')
print(repr(text), '\n')
print(re.search(r'problem: "(.*?)"(\n|.\n)sol', text, re.DOTALL))
if not re.search(r'problem: "(.*?)"(\n|.\n)sol', text, re.DOTALL):
    # If not, enclose it in double quotes
    text = re.sub(r'(problem: )(.*)((\n|.\n)sol)', r'\1"\2"\3', text)
text

'to_drop: false\nproblem: "Is the following statement true or false? \nZero is the smallest natural number".\nsolution:\n  - "The given statement is **False**. \nThe set of natural numbers is defined as \n$$\\mathbb{N} = \\{ 1, 2, 3, \\dots \\}$$. \nTherefore, 1 is the smallest natural number."\n' 

<re.Match object; span=(15, 109), match='problem: "Is the following statement true or fals>


'to_drop: false\nproblem: "Is the following statement true or false? \nZero is the smallest natural number".\nsolution:\n  - "The given statement is **False**. \nThe set of natural numbers is defined as \n$$\\mathbb{N} = \\{ 1, 2, 3, \\dots \\}$$. \nTherefore, 1 is the smallest natural number."\n'

In [360]:
text = 'to_drop: false\\nproblem: Which of the following number is prime? $51$\\nsolution: -  A prime number is a natural number greater than 1 that is not a product of two smaller natural numbers. \\n           -  $51$ can be written as $3 \\\\times 17$\\n           -  Therefore, $51$ is a composite number.\\n```","input_tokens":356.0,"output_tokens":88.0}\n'

text = re.sub(r'\\{1,}n', '\n', text)

x, y = re.search(r'to_drop: (.*)\n', text).span()
print(text[x:y-1])
print(yaml.full_load(text[x:y-1]), '\n')

# x, y = re.search(r'problem: (.*)\nsol', text, re.DOTALL).span()
print(repr(text), '\n')
print(re.search(r'problem: "(.*?)"(\n|.\n)sol', text, re.DOTALL))
if not re.search(r'problem: "(.*?)"(\n|.\n)sol', text, re.DOTALL):
    # If not, enclose it in double quotes
    text = re.sub(r'(problem: )(.*)((\n|.\n)sol)', r'\1"\2"\3', text)
text
# print(text[x:y-4])
# print(yaml.full_load(text[x:y-4]), '\n')

# x, y = re.search(r'solution:\n (.*)', text, re.DOTALL).span()
# print(repr(text[x:y]), '\n')
# print(yaml.full_load(text[x:y].replace("\\", "\\\\")), '\n')

to_drop: false
{'to_drop': False} 

'to_drop: false\nproblem: Which of the following number is prime? $51$\nsolution: -  A prime number is a natural number greater than 1 that is not a product of two smaller natural numbers. \n           -  $51$ can be written as $3 \\\\times 17$\n           -  Therefore, $51$ is a composite number.\n```","input_tokens":356.0,"output_tokens":88.0}\n' 

None


'to_drop: false\nproblem: "Which of the following number is prime? $51$"\nsolution: -  A prime number is a natural number greater than 1 that is not a product of two smaller natural numbers. \n           -  $51$ can be written as $3 \\\\times 17$\n           -  Therefore, $51$ is a composite number.\n```","input_tokens":356.0,"output_tokens":88.0}\n'

In [383]:
text = '''to_drop: false
problem: "Is the following statement true or false? <br> Zero is the smallest natural number."
solution: |-
  The given statement is **False**. <br>
  The set of natural numbers is defined as <br>
  $$\mathbb{N} = \{ 1, 2, 3, \dots \}$$. <br>
  Therefore, 1 is the smallest natural number.'''
text = re.sub(r'- +', ' -', text)
text = re.sub(r' +', ' ', text)
print(repr(text), '\n')
print(text, '\n')
yaml.full_load(text)

'to_drop: false\nproblem: "Is the following statement true or false? <br> Zero is the smallest natural number."\nsolution: |-\n The given statement is **False**. <br>\n The set of natural numbers is defined as <br>\n $$\\mathbb{N} = \\{ 1, 2, 3, \\dots \\}$$. <br>\n Therefore, 1 is the smallest natural number.' 

to_drop: false
problem: "Is the following statement true or false? <br> Zero is the smallest natural number."
solution: |-
 The given statement is **False**. <br>
 The set of natural numbers is defined as <br>
 $$\mathbb{N} = \{ 1, 2, 3, \dots \}$$. <br>
 Therefore, 1 is the smallest natural number. 



{'to_drop': False,
 'problem': 'Is the following statement true or false? <br> Zero is the smallest natural number.',
 'solution': 'The given statement is **False**. <br>\nThe set of natural numbers is defined as <br>\n$$\\mathbb{N} = \\{ 1, 2, 3, \\dots \\}$$. <br>\nTherefore, 1 is the smallest natural number.'}

In [380]:
processed_data = []
error_at = []

with open(interim_fname, 'r') as f:
    raw_data = f.readlines()

for idx, d in tqdm(enumerate(raw_data), total=len(raw_data)):
    d = json.loads(d)
    output_text = d['text']

    if output_text is None or len(output_text.strip()) == 0:
        continue

    output_text = output_text.replace('```yaml\n', '').replace('```', '').replace('\\\\newline', '').replace('\\\\\n', '\n')
    output_text = output_text.replace(r'\\(', '$').replace(r'\\)', '$').replace("\\", "\\\\")
    output_text = re.sub(r'\\{1,}n', '\n', output_text)
    output_text = re.sub(r'- +', ' -', output_text)
    output_text = re.sub(r' +', ' ', output_text)


    # x, y = re.search(r'to_drop: (.*)\n', text).span()
    # print(text[x:y-1])
    # print(yaml.full_load(text[x:y-1]), '\n')

    # x, y = re.search(r'problem: (.*)\nsol', text, re.DOTALL).span()
    # print(text[x:y-4])
    # print(yaml.full_load(text[x:y-4]), '\n')

    # x, y = re.search(r'solution:\n (.*)', text, re.DOTALL).span()
    # print(repr(text[x:y]), '\n')
    # print(yaml.full_load(text[x:y].replace("\\", "\\\\")), '\n')
    # if not re.search(r'problem: "(.*?)"(\n|.\n)sol', output_text, re.DOTALL):
    #     # If not, enclose it in double quotes
    #     output_text = re.sub(r'(problem: )(.*)((\n|.\n)sol)', r'\1"\2"\3', output_text)
    
    # output_text = re.sub(r'\\\\', r'\\', output_text)

    try:
        output_text = yaml.full_load(output_text)
    except Exception:
        # print(f"Error at {d['exp_id']} | {idx}")
        error_at.append(d['exp_id'])
        # print(f'\nRAW Text - {idx}\n{repr(output_text)}' )
        if not re.search(r'problem: "(.*?)"(\n|.\n)sol', output_text, re.DOTALL):
            # If not, enclose it in double quotes
            output_text = re.sub(r'(problem: )(.*)((\n|.\n)sol)', r'\1"\2"\3', output_text)
        # continue
        # print(output_text)
        output_text = re.sub(r'\\\\', r'\\', output_text)

        # print(f"\nBefore Loading:\n{repr(output_text)}")

        try:
            print(f"\nBefore Loading:\n{repr(output_text)}")
            output_text = yaml.full_load(fr"{output_text}")
            print(f"\nAfter Loading:\n{repr(output_text)}")
        except Exception as e:
            print(f"Error at {d['exp_id']} | {idx}")
            print(e)
            break
        continue

    if output_text['to_drop']:
        continue
            
    try:
        processed_data.append({
            'exp_id': d['exp_id'],
            'problem': output_text['problem'],
            'solution': output_text['solution'],
            'is_blocked': d['is_blocked'],
        })
    except KeyError:
        print(f"Error at {d['exp_id']} | {idx}")
        print(output_text)
        break


  0%|          | 0/4303 [00:00<?, ?it/s]


Before Loading:
'to_drop: false\nproblem: "Is the following statement true or false? \nZero is the smallest natural number."\nsolution:\n -"The given statement is **False**. \nThe set of natural numbers is defined as \n$$\\mathbb{N} = \\{ 1, 2, 3, \\dots \\}$$. \nTherefore, 1 is the smallest natural number."\n'
Error at Exp0016 | 15
while scanning a simple key
  in "<unicode string>", line 6, column 1:
    The set of natural numbers is de ... 
    ^
could not find expected ':'
  in "<unicode string>", line 7, column 1:
    $$\mathbb{N} = \{ 1, 2, 3, \dots ... 
    ^


In [381]:
raw_data[15]

'{"id":16,"exp_id":"Exp0016","is_blocked":true,"text":"```yaml\\nto_drop: false\\nproblem: \\"Is the following statement true or false? \\\\\\\\\\nZero is the smallest natural number.\\"\\nsolution:\\n  - \\"The given statement is **False**. \\\\\\\\\\nThe set of natural numbers is defined as \\\\\\\\\\n$$\\\\mathbb{N} = \\\\{ 1, 2, 3, \\\\dots \\\\}$$. \\\\\\\\\\nTherefore, 1 is the smallest natural number.\\"\\n```","input_tokens":322.0,"output_tokens":90.0}\n'

In [151]:
len(error_at)

3041

### Throw back to LLM

In [392]:
rewrite_prompt = """ Write only the solution as numbered steps with proper latex.
Fix the latex if necessary.

```yaml
{initial_output}
```

If you can find more efficient solution, please present that otherwise use the same solution."""

In [393]:
with open(interim_fname, 'r') as f:
    raw_data = f.readlines()

In [401]:
raw_data[0]

'{"id":2,"exp_id":"Exp0002","is_blocked":true,"text":"```yaml\\nto_drop: false\\nproblem: \\"Which is the smallest whole number?\\"\\nsolution:\\n  - \\"The smallest whole number is $0$ (zero).  Whole numbers include all the natural numbers along with $0$.\\"\\n```","input_tokens":318.0,"output_tokens":53.0}\n'

In [408]:

exp_ids = []
initial_outputs = []
prompts = []
for d in tqdm(raw_data):
    d = json.loads(d)
    output_text = d['text']
    prompt = rewrite_prompt.format(initial_output=output_text)

    exp_ids.append(d['exp_id'])
    initial_outputs.append(output_text)
    prompts.append(prompt)
    

  0%|          | 0/4303 [00:00<?, ?it/s]

In [409]:
rewrite_df = pd.DataFrame({
    'id': exp_ids,
    'initial_output': initial_outputs,
    'prompt': prompts
})
rewrite_df

,id,initial_output,prompt
0,Exp0002,"```yaml\nto_drop: false\nproblem: ""Which is th...",Write only the solution as numbered steps wit...
1,Exp0001,"```yaml\nto_drop: false\nproblem: ""Write the t...",Write only the solution as numbered steps wit...
2,Exp0000,"```yaml\nto_drop: false\nproblem: ""Write the n...",Write only the solution as numbered steps wit...
3,Exp0004,"```yaml\nto_drop: false\nproblem: ""Write the s...",Write only the solution as numbered steps wit...
4,Exp0003,```yaml\nto_drop: false\nproblem: How many who...,Write only the solution as numbered steps wit...
...,...,...,...
4298,Exp3550,"```yaml\nto_drop: false\nproblem: ""Prove that ...",Write only the solution as numbered steps wit...
4299,Exp4096,"```yaml\nto_drop: false\nproblem: ""For given v...",Write only the solution as numbered steps wit...
4300,Exp4114,"```yaml\nto_drop: false\nproblem: ""If $\\vec{a...",Write only the solution as numbered steps wit...
4301,Exp4115,```yaml\nto_drop: false\nproblem: Find a unit ...,Write only the solution as numbered steps wit...


In [410]:
# # interim_fname = f'interim_results_Preprocessing_input_data_{pd.to_datetime("now").strftime("%Y%m%d_%H%M%S")}.jsonl'
# interim_fname = 'interim_results_Preprocessing_input_data_gemini_flash_2.jsonl'
# print(f"Interim Results File: {interim_fname}")

# if not os.path.exists(interim_fname):
#     with jsonlines.open(interim_fname, 'w') as writer:
#         writer.write_all([])

In [434]:
interim_fname = 'interim_results_Preprocessing_input_data_gemini_flash_2.jsonl'
with jsonlines.open(interim_fname, 'r') as reader:
    interim_results_df = pd.DataFrame(reader)

if len(interim_results_df) == 0:
    interim_results_df = pd.DataFrame(columns=['id', 'is_blocked', 'text', 'input_tokens', 'output_tokens', 'DropReason'])

In [435]:
interim_results_df.text.isna().sum()

0

In [433]:
interim_results_df = interim_results_df[interim_results_df.text.notna()].copy(deep=True)
interim_results_df.to_json(interim_fname, orient='records', lines=True, index=False)

In [436]:
# intermedate_df = asyncio.run(get_raw_output_from_llm_v2(chucked_df, interim_results_df))
intermedate_df = asyncio.run(
    get_raw_output_from_llm(
        df=rewrite_df,
        interim_results_df=interim_results_df,
        interim_fname=interim_fname,
        system_instruction='',
        model_name='gemini-1.5-flash-preview-0514',
        # model_name='gemini-1.0-pro-002',
        prompt_col='prompt',
        save_every=10,
        rpm=180,
    )
)

Creating Tasks:   0%|          | 0/4303 [00:00<?, ?it/s]

22 Tasks Created


Waiting for Tasks:: 100%|███████████████████████████████████████████| 22/22 [00:11<00:00,  1.94it/s]

Number of Safety Blocks: 22
Number of Input not processed: 0


## Create Training Data

In [147]:
data = {}

with open("interim_results_Preprocessing_input_data_gemini_flash.jsonl", 'r') as f:
    raw_data = f.readlines()

for d in tqdm(raw_data):
    current_row = json.loads(d)
    text = current_row['text']

    if 'Exp0281' == current_row['exp_id']:
        print(repr(text))
        break

    if 'to_drop: true' in text:
        continue

    if 'figure' in text:
        continue

    try:
        x, y = re.search(r'problem: (.*)\nsol', text, re.DOTALL).span()
        problem = text[x:y-4].strip()
        problem = problem.replace('\\\\newline', '')
    except AttributeError:
        print(text)
        break

    # Remove excess spaces
    problem = re.sub(r'\\{1,}n', '\n', problem)
    problem = re.sub(r' +', ' ', problem)
    problem = re.sub(r'(problem:)(\s+)(\S)', r'\1 \3', problem, re.DOTALL)
    problem = problem.replace('"', '')

    if problem[9] != '"':
        problem = problem[:9] + '"' + problem[9:]

    if problem[-1] != '"':
        problem = problem + '"'

    # Replace single backslash with double backslash
    problem = problem.replace("\\", "\\\\")
    problem = re.sub(r'\\{2,}', r'\\\\', problem)

    # Clean text
    problem = problem.replace('\\~\\', '')
    problem = re.sub(r'\\+\n', r'\n', problem)
    problem = re.sub(r'\\{1,} ', r' ', problem)
    problem = re.sub(r' +', ' ', problem)

    # Replace "\\(" and "\\)" with "$" and "\\[" and "\\]" with "$$"
    problem = problem.replace(r'\\(', '$').replace(r'\\)', '$').replace(r'\\[', '$$').replace(r'\\]', '$$')

    try:
        problem = yaml.full_load(rf"{problem}")
    except Exception:
        print(repr(f"{current_row['exp_id']} | {text[x:y]}"), '\n')
        print(repr(problem), '\n')
        print(yaml.full_load(fr"{problem}"), '\n')
        break

    data[current_row['exp_id']] = {
        'problem': problem['problem'],
        'solution': []
    }

  0%|          | 0/4303 [00:00<?, ?it/s]

'```yaml\nto_drop: true\nproblem: ""\nsolution: []\n```'


In [139]:
from copy import deepcopy
data_copy = deepcopy(data)

In [158]:
with open("interim_results_Preprocessing_input_data_gemini_flash_2.jsonl", 'r') as f:
    raw_data = f.readlines()

for d in tqdm(raw_data):
    current_row = json.loads(d)
    exp_id = current_row['exp_id']
    solution = current_row['text']

    if solution is None:
        continue

    if exp_id not in data_copy:
        # print(f"Dropping {exp_id}")
        continue

    # Cleanup text and latex
    try:
        solution = solution.replace('\\\\newline', '')
        solution = re.sub(r'\\{1,}n', '\n', solution)
        solution = re.sub(r'\\+\n', r'\n', solution)
        solution = solution.replace('"', '')
        solution = solution.replace("\\", "\\\\")
        solution = re.sub(r'\\{2,}', r'\\\\', solution)
        solution = re.sub(r'\\{1,} ', r' ', solution)
        solution = re.sub(r'\n', ' ', solution)
        solution = re.sub(r' +', ' ', solution)
    except AttributeError:
        print(current_row)
        break

    steps = re.split(r'\d+\. ', solution)

    if steps[0] == '':
        steps = steps[1:]
    
    steps = [s.strip() for s in steps]

    data_copy[exp_id]['solution'] = steps

  0%|          | 0/4303 [00:00<?, ?it/s]

In [162]:
data_copy['Exp0010']

{'problem': 'Write the predecessor of $208090$.',
 'solution': ['The predecessor of a number is the number that comes just before it.',
  'Therefore, the predecessor of $208090$ is $208090-1 = 208089$.']}

In [163]:
with open("final_data.json", 'w') as f:
    json.dump(data_copy, f, indent=4)

## Create JSON for Chat Template

In [3]:
with open("final_data.json", 'r') as f:
    data = json.load(f)

with open('chat_template.txt', 'r') as f:
    chat_template = f.read()

In [8]:
info_df = pd.read_json(os.path.join(BASE_FOLDER, 'preprocess_input.jsonl'), lines=True)
info_df

,id,problem,solution,prompt,concept,chapter_name
0,Exp0000,Write the next three natural numbers after 10999.,"10,999+1 = 11,000\n11,000 + 1 = 11,001\n11,001...",<math_problem>\nWrite the next three natural n...,Concept for Natural Numbers,Whole Numbers
1,Exp0001,Write the three whole numbers occurring just b...,Three whole numbers occurring just before 1000...,<math_problem>\nWrite the three whole numbers ...,Concept for Whole Numbers,Whole Numbers
2,Exp0002,Which is the smallest whole number?,0 (zero) is the smallest whole number. All the...,<math_problem>\nWhich is the smallest whole nu...,Concept for Whole Numbers,Whole Numbers
3,Exp0003,How many whole numbers are there between 32 an...,Whole numbers between 32 and 53 = 20 (53 − 32 ...,<math_problem>\nHow many whole numbers are the...,Concept for Whole Numbers,Whole Numbers
4,Exp0004,Write the successor of 2440701.,The successor of 2440701 is\n2440701 + 1 = 244...,<math_problem>\nWrite the successor of 2440701...,Successor and Predecessor of Whole Number,Whole Numbers
...,...,...,...,...,...,...
4321,Exp4321,If each element of a second order determinant ...,The total number of determinants of second ord...,<math_problem>\nIf each element of a second or...,Independent Events,Probability
4322,Exp4322,An electronic assembly consists of two subsyst...,Let the event in which A fails and B fails be ...,<math_problem>\nAn electronic assembly consist...,Properties of Conditional Probability,Probability
4323,Exp4323,If A and B are two events such that P (A) ≠ 0 ...,"P (A) ≠ 0 and P(B|A) = 1\n\nThus, the correct ...",<math_problem>\nIf A and B are two events such...,Properties of Conditional Probability,Probability
4324,Exp4324,"If P (A|B) > P (A), then which of the followin...","\nThus, the correct answer is C.","<math_problem>\nIf P (A|B) > P (A), then which...",Properties of Conditional Probability,Probability


In [4]:
gemma_tokenizer = AutoTokenizer.from_pretrained("google/gemma-1.1-2b-it")
gemma_tokenizer.chat_template = chat_template

In [5]:
system_prompt = "Your are a high school student and you are appearing for your math exam.\nYou can solve given math problems with step-by-step mathematical reasoning as well as using sympy for calculations."

In [6]:
FOLDER_NAME = 'Collected_Math_Problems'
if not os.path.exists(os.path.join(ROOT_DATA_DIR, FOLDER_NAME)):
    os.makedirs(os.path.join(ROOT_DATA_DIR, FOLDER_NAME))

FOLDER_PATH = os.path.join(ROOT_DATA_DIR, FOLDER_NAME)
os.path.exists(FOLDER_PATH)

True

In [7]:
current_data = data["Exp3102"]
current_data

{'problem': 'How many words, with or without meaning, each of 2 vowels and 3 consonants can be formed from the letters of the word DAUGHTER?',
 'solution': ['The word DAUGHTER has 3 vowels (A, U, E) and 5 consonants (D, G, H, T, R).',
  'We need to select 2 vowels out of 3, which can be done in $^3C_2 = 3$ ways.',
  'We also need to select 3 consonants out of 5, which can be done in $^5C_3 = 10$ ways.',
  'Therefore, the total number of combinations of 2 vowels and 3 consonants is 3 × 10 =',
  '',
  'Each of these 30 combinations can be arranged in 5! = 120 ways.',
  'Hence, the total number of words that can be formed is 30 × 120 =',
  '']}

In [185]:

for exp_id, current_data in tqdm(data.items()):
    maths_problem_message = [
        {
            'role': 'system',
            'content': system_prompt
        }
    ]

    maths_problem_message.append(
        {
            'role': 'problem',
            'content': current_data['problem']
        }
    )

    for step in current_data['solution']:
        maths_problem_message.append(
            {
                'role': 'step',
                'content': [
                    {
                        'role': 'text',
                        'content': step
                    }
                ]
            }
        )

    with open(os.path.join(FOLDER_PATH, f"{exp_id}.json"), 'w') as f:
        json.dump(maths_problem_message, f, indent=4)

  0%|          | 0/4198 [00:00<?, ?it/s]

## Playground

In [9]:
FOLDER_NAME = 'Collected_Math_Problems'
FOLDER_PATH = os.path.join(ROOT_DATA_DIR, FOLDER_NAME)

fnames = os.listdir(FOLDER_PATH)
os.path.exists(FOLDER_PATH)

True

In [10]:
a = [1, 3]
a.insert(1, 2)
a

[1, 2, 3]

In [11]:
from IPython.display import display, Markdown

In [12]:
idx = 0
with open(os.path.join(FOLDER_PATH, fnames[idx]), 'r') as f:
    data = json.load(f)
data

[{'role': 'system',
  'content': 'Your are a high school student and you are appearing for your math exam.\nYou can solve given math problems with step-by-step mathematical reasoning as well as using sympy for calculations.'},
 {'role': 'problem',
  'content': 'In a box containing 100 bulbs, 10 are defective. The probability that out of a sample of 5 bulbs, none is defective is (A) $10^{-1}$ (B) $(\\frac{1}{2})^5$ (C) $(\\frac{9}{10})^5$ (D) $\\frac{9}{10}$'},
 {'role': 'step',
  'content': [{'role': 'text',
    'content': 'The probability of selecting a defective bulb is $p=\\\\frac{10}{100}=\\\\frac{1}{10}$.'}]},
 {'role': 'step',
  'content': [{'role': 'text',
    'content': 'Therefore, the probability of selecting a non-defective bulb is $1-p = \\\\frac{9}{10}$.'}]},
 {'role': 'step',
  'content': [{'role': 'text',
    'content': 'Since the selection of bulbs is independent, the probability that out of a sample of 5 bulbs, none are defective is $(\\\\frac{9}{10})^5$.'}]}]

In [13]:
display(Markdown(data[1]['content']))

In a box containing 100 bulbs, 10 are defective. The probability that out of a sample of 5 bulbs, none is defective is (A) $10^{-1}$ (B) $(\frac{1}{2})^5$ (C) $(\frac{9}{10})^5$ (D) $\frac{9}{10}$

In [14]:
for content in data[2:]:
    if content['role'] == 'step':
        display(Markdown(re.sub(r'\\\\', r'\\', content['content'][0]['content'])))

The probability of selecting a defective bulb is $p=\frac{10}{100}=\frac{1}{10}$.

Therefore, the probability of selecting a non-defective bulb is $1-p = \frac{9}{10}$.

Since the selection of bulbs is independent, the probability that out of a sample of 5 bulbs, none are defective is $(\frac{9}{10})^5$.

In [15]:
from sympy import Rational
from sympy.parsing.latex import parse_latex

In [16]:
print(1 - Rational(5, 6))

1/6


In [17]:
print((Rational("0.05") * Rational(1, 2)) / (Rational("0.05")* Rational(1, 2) + Rational("0.0025") * Rational(1, 2)))

20/21


In [18]:
from sympy import binomial, Rational

n = 10
p = Rational(1, 6)

probabilities = []
for k in range(2):
    probability = binomial(n, k) * p**k * (1 - p)**(n - k)
    print(binomial(n, k), p**k, (1 - p)**(n - k))
    probabilities.append(probability)

print("Probabilities:")
for k, probability in enumerate(probabilities):
    print(f"P(X={k}) = {probability}")

sum(probabilities)

1 1 9765625/60466176
10 1/6 1953125/10077696
Probabilities:
P(X=0) = 9765625/60466176
P(X=1) = 9765625/30233088


9765625/20155392

In [19]:
125 / 648, 1953125 / 10077696

(0.19290123456790123, 0.19380669946781487)

In [20]:
Rational(9765625, 60466176), Rational(1953125, 60466176)

(9765625/60466176, 1953125/60466176)

In [21]:
from sympy import binomial, Rational

p = Rational(2, 3)
n = 6  # Number of throws
ks =  range(0, 4)

probabilities = [binomial(n, k) * p**k * (1 - p)**(n-k) for k in ks]

print(1 - sum(probabilities)) 

496/729


In [22]:
496/729, probabilities, 1 -sum(probabilities), 1+12+60+160

(0.6803840877914952, [1/729, 4/243, 20/243, 160/729], 496/729, 233)

In [23]:
from scipy.stats import binom

# Define the parameters
n = 10
p = 0.6
k = range(4)  # k values from 0 to 6

# Calculate the probabilities for each k value
probabilities = binom.pmf(k, n, p)

# Sum the probabilities
total_probability = 1 - sum(probabilities)

print(total_probability)

0.9452381184


In [24]:
from scipy.stats import binom

# Define the parameters
n = 10
p = 0.6
k = 3

# Calculate the probabilities for each k value
probability = 1 - binom.cdf(k, n, p)

print(probability)

0.9452381183999999


In [25]:
from sympy import Symbol, solve, Rational

n = Symbol('n')
equation = 1 - Rational(1, 2)**n > 0.9
solution = solve(equation, n)
solution

2.30258509299405/log(2) < n

In [26]:
solution.evalf(1)

3.0 < n

In [27]:
from fractions import Fraction

expectation = Fraction(1, 6) * 1 + \
              Fraction(5, 6) * Fraction(1, 6) * 0 + \
              Fraction(5, 6) * Fraction(5, 6) * Fraction(1, 6) * (-1) + \
              Fraction(5, 6) * Fraction(5, 6) * Fraction(5, 6) * (-3)

print(expectation)

-91/54


In [28]:
from sympy import Rational

# Define values
value = Rational(52 - 33, 1) + 1
print(value)

20


In [29]:
multiples_of_9 = []
n = 1

while 9 * n <= 100:
  multiples_of_9.append(9 * n)
  n += 1

print(multiples_of_9)

[9, 18, 27, 36, 45, 54, 63, 72, 81, 90, 99]


In [30]:
import sympy

In [40]:
from sympy import divisors
print(divisors(24))

[1, 2, 3, 4, 6, 8, 12, 24]


In [45]:
from sympy import symbols, Eq, solve

# Define the variables
x, y = symbols('x y')

# Define the equations
eq1 = Eq(2*x + 3*y, 11)
eq2 = Eq(2*x - 4*y, -24)

# Solve the system of equations
solution = solve((eq1, eq2), (x, y))

print(solution)

{x: -2, y: 5}


In [49]:
from math import gcd
print(gcd(70, 105, 175))

35


In [65]:
from math import gcd

HCF = gcd(18, 48)
print(HCF)

6


In [18]:
from sympy import Rational

value = Rational(-2, 13) * Rational(7, 1)
print(value)

-14/13


In [38]:
def find_repeating_block(numerator, denominator):
    remainder = numerator % denominator
    seen_remainders = {remainder: 0}
    decimal_expansion = ""

    while True:
        remainder *= 10
        digit, remainder = divmod(remainder, denominator)
        decimal_expansion += str(digit)

        if remainder in seen_remainders:
            start = seen_remainders[remainder]
            return decimal_expansion[start:]

        seen_remainders[remainder] = len(decimal_expansion)

# Test the function
repeating_block = find_repeating_block(1, 17)
print(f"Repeating block of digits: {repeating_block}")
print(f"Length of repeating block: {len(repeating_block)}")

Repeating block of digits: 0588235294117647
Length of repeating block: 16


In [70]:
data = [6, 7, 10, 12, 13, 4, 8, 12]
mean = sum(data) / len(data)
print(mean)

9.0


In [78]:
from sympy import sqrt, Rational

variance = Rational(35, 6)
standard_deviation = sqrt(variance)

print(standard_deviation)

sqrt(210)/6


In [88]:
from sympy import binomial, Rational

def binomial_pmf(n, k, p, q):
  return binomial(n, k) * (p**k) * (q**(n-k))

P_X_4 = binomial_pmf(5, 4, Rational(1, 3), Rational(2, 3))
P_X_5 = binomial_pmf(5, 5, Rational(1, 3), Rational(2, 3))


print(P_X_4, P_X_5)

10/243 1/243


In [1]:
# print((1/6 * 1) + (5/36 * 0) + (25/216 * -2) + (125/216 * -3))
from sympy import Rational

expectation = Rational(1, 6) * 1 + Rational(5, 36) * 0 + Rational(25, 216) * -2 + Rational(125, 216) * -3

print(expectation)

-389/216


In [8]:
from sympy import Rational

value = (0.28 * Rational(1, 2)) / ((0.28 * Rational(1, 2)) + (0.3 * Rational(1, 2)))
value2 = (Rational("0.28") * Rational(1, 2)) / ((Rational("0.28") * Rational(1, 2)) + (Rational("0.3") * Rational(1, 2)))
print(value)
print(value2)

0.482758620689655
14/29


In [2]:
from sympy import Rational, binomial

value = Rational(5, 2) * (Rational(1, 6)) ** 2 * (Rational(5, 6)) ** 3
value2 = binomial(5, 2) * (Rational(1, 6)) ** 2 * (Rational(5, 6)) ** 3

print(value, value2)

625/15552 625/3888


In [26]:
from sympy import symbols, solve, Eq

k, l, x = symbols("k l x")
eq = Eq(k * x**2 - 2 * k * x + l, 4)
x_coords = solve(eq, x)
print(x_coords)

[(k - sqrt(k*(k - l + 4)))/k, (k + sqrt(k*(k - l + 4)))/k]


In [3]:
import math


def quadratic_formula(a, b, c):
    x1 = (-b + math.sqrt(b**2 - 4 * a * c)) / (2 * a)
    x2 = (-b - math.sqrt(b**2 - 4 * a * c)) / (2 * a)
    return x1, x2


k = 1  # Replace with the actual value of k
l = 2  # Replace with the actual value of l

x_A, x_B = quadratic_formula(k, -2 * k, l - 4)

print(f"x_A: {x_A}")
print(f"x_B: {x_B}")

x_A: 2.732050807568877
x_B: -0.7320508075688772


In [4]:
from langchain_community.utilities.python import PythonREPL

python_repl = PythonREPL()

In [5]:
python_repl.run("print(1+2)")

Python REPL can execute arbitrary code. Use with caution.


'3\n'

In [2]:
from interpreter import interpreter

In [9]:
python_repl.run("""import math

k = 1  # Replace with the actual value of k
l = 2  # Replace with the actual value of l

x_A = (-b + math.sqrt(b**2 - 4 * a * c)) / (2 * a)
x_B = (-b - math.sqrt(b**2 - 4 * a * c)) / (2 * a)
                
print(f"x_A: {x_A}")
print(f"x_B: {x_B}")""")

'NameError("name \'b\' is not defined")'

In [7]:
interpreter.computer.run("python", """import math


def quadratic_formula(a, b, c):
    x1 = (-b + math.sqrt(b**2 - 4 * a * c)) / (2 * a)
    x2 = (-b - math.sqrt(b**2 - 4 * a * c)) / (2 * a)
    return x1, x2


k = 1  # Replace with the actual value of k
l = 2  # Replace with the actual value of l

x_A, x_B = quadratic_formula(k, -2 * k, l - 4)

print(f"x_A: {x_A}")
print(f"x_B: {x_B}")""")[0]['content'].strip()

'x_A: 2.732050807568877\nx_B: -0.7320508075688772'

In [1]:
from sympy import symbols, Eq, solve

# Define the variables
x, y, z = symbols('x y z')

# Define the equations
eq1 = Eq(x * y * z, 1)
eq2 = Eq(x + 1/z, 5)
eq3 = Eq(y + 1/x, 29)

# Solve the system of equations
solution = solve((eq1, eq2, eq3), (x, y, z))

print(solution)

[(1/5, 24, 5/24)]


In [9]:
from sympy import symbols, Eq, solve

# Define symbols
b, x, y = symbols('b x y')

# Define the equation
equation = Eq(3*x + b, y)

# Substitute x = 2 into the equation
equation = equation.subs(x, 4).subs(y, 5)

# Solve for a
solution = solve(equation, b)

# Print the solution
print(solution)

[-7]


In [10]:
import numpy as np
np.linalg.det(np.array([[2, 1], [7, -3]]))

-13.0

In [5]:
from sympy import Rational
print(Rational(2**4 + 2*(5**2), 6))

11


In [24]:
from sympy import binomial

result = binomial(14, 11)
print(result)

364


In [55]:
import math

def f(x):
  return x / 100 - math.sin(x)

def find_roots(a, b, tolerance=1e-6):
  """Finds the number of roots of f(x) in the interval [a, b].

  Args:
    a: The left endpoint of the interval.
    b: The right endpoint of the interval.
    tolerance: The tolerance for considering a value to be a root.

  Returns:
    The number of roots in the interval.
  """
  count = 0
  x = a
  while x <= b:
    if abs(f(x)) < tolerance:  # Found a root
      count += 1
      # Skip ahead to avoid counting the same root multiple times
      x += 0.1 
    else:
      x += tolerance
  return count

# Count roots in each interval [2*pi*k, 2*pi*(k+1)]
total_roots = 0
for k in range(16):
  a = 2 * math.pi * k
  b = 2 * math.pi * (k + 1)
  roots_in_interval = find_roots(a, b)
  total_roots += roots_in_interval

print(f"Total number of solutions: {total_roots}")

Total number of solutions: 32


In [63]:
25/9 - 2.777777777778 < 1e-6

True